In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import Counter

In [ ]:
# FUNCTIONS

def path(folder,n_file):
    n_file_str = str(n_file).zfill(3)
    file_path = str(r"C:\Users\rjnar\Documents\NIP\BS-Thesis\III - EEG Analysis\\" + str(folder) + r"\\" + str(folder) + n_file_str + '.txt')
    return file_path

def extract(file_path):
    with open(file_path, 'r') as file:
        data = [ int(line.strip()) for line in file]
    return data

# GLOBAL FORMATS
def analyze_folder(folder):                         # Main Function
    data = []                                       # extracts the folder into a csv
    for i in range(1,101):
        name = path(folder,i)
        data.append( extract(name) )
    return data


In [ ]:
# Conventions, Customs, Data

y_modifiers = []
for i in range(100):
    y_modifiers.append(i*200)
print(y_modifiers)

folder_names = {
    "Z":"Z",
    "O":"O",
    "N":"N",
    "F":"F",
    "S":"S"
}

folder_mapping = {
    "Z":"A",
    "O":"B",
    "N":"C",
    "F":"D",
    "S":"E"
}

# Time index
time_index = np.arange(4097)
print(time_index)

# Folders
folder_list = ["Z","O","N","F","S"]


In [ ]:

def EEGram(folder):
    data = np.array( analyze_folder(folder_names[folder])  )           # Data is a 2D ARRAY

    row_variances = np.var(data, axis=1)
    sorted_indices = np.argsort(row_variances)
    data = data[sorted_indices]
    
    run = 0
    
    fig , axs = plt.subplots(1,1, figsize=(40,100))
    fig.suptitle(f"EEGram for Set {folder_names[folder]}({folder_mapping[folder]}), Rwen Narvarte", fontsize=128)
    axs.set_ylim(-100,20000)
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    for i in range(100):
        series = data[run]                       # Converted the list to a 1D array for faster runtime
        mean_series = round(np.mean(series))                # Get mean of series
        dy = y_modifiers[i] - mean_series                   # distance between correct height vs current height
        corrected_series = series + dy                      # correcting the height

        corrected_mean = round(np.mean(corrected_series))
        corrected_series_mean = [corrected_mean] * len(corrected_series)
        
        axs.plot(time_index, corrected_series_mean, color='black')
        axs.plot(time_index, corrected_series, color='tab:blue')
        
        run += 1

    plt.savefig(f'{folder_mapping[folder]}_EEGram_sorted', dpi=72)

In [ ]:
def varianceplt(folder):
    data = np.array( analyze_folder(folder_names[folder])  )           # Data is a 2D ARRAY

    variance = np.var(data, axis=1)
    variance = sorted(variance, reverse = True)
    return variance



In [ ]:
plt.figure(figsize=(12, 8))  # Create a single figure for all data

for folder in folder_list:
    # Calculate variance for the current folder
    variance = varianceplt(folder_names[folder])
    
    # Create a scatter plot for the current folder
    plt.scatter(
        [i + 1 for i in range(len(variance))],  # Ensure x-values start from 1 (log(0) undefined)
        variance,
        label=f"Folder: {folder}"
    )

# Set log-log scale
plt.xscale('log')
plt.yscale('log')

# Add titles, labels, and legend
plt.title("Variance Plots for All Folders (Log-Log Scale)")
plt.xlabel("Row Index (Log Scale)")
plt.ylabel("Variance (Log Scale)")
plt.legend()
plt.grid(visible=True, which="both", linestyle="--", linewidth=0.5)

# Show the plot
plt.show()


### FRD